# Convert CSV to Arrow with chDB

chDB runs the same `SELECT ... INTO OUTFILE ... FORMAT Arrow` as `clickhouse local`, in-process from Python. Run `./generate.sh` first to create `data/events.csv`.

Companion article: https://clickhouse.com/resources/engineering/convert-csv-to-arrow

In [ ]:
import os
import chdb

os.chdir("data")

## 1. Convert CSV -> Arrow in one line

In [ ]:
chdb.query("SELECT * FROM file('events.csv') INTO OUTFILE 'events_py.arrow' TRUNCATE FORMAT Arrow")
print("written events_py.arrow")

## 2. Prove it: row count + the types embedded in the Arrow schema

In [ ]:
print("rows:", chdb.query("SELECT count() FROM file('events_py.arrow')", "CSV"), end="")
print(chdb.query("DESCRIBE file('events_py.arrow')", "TabSeparated"), end="")

## 3. Query the Arrow file straight back as a DataFrame

In [ ]:
df = chdb.query(
    "SELECT country, count() AS events, round(sum(amount),2) AS amount "
    "FROM file('events_py.arrow') GROUP BY country ORDER BY amount DESC LIMIT 5",
    "DataFrame",
)
print(df.to_string(index=False))